# Preiskovanje v igrah

Na prejšnjem predavanju smo spoznali preiskovalne algoritme, ki jih uporabljamo za reševanje problemov, predstavljenih kot prostor stanj možnih rešitev. Tak problem običajno vključuje množico stanj, začetno stanje, množico dovoljenih prehodov (funkcijo prehodov, ki določa tudi njihovo ceno) med stanji in test za preverjanje ali je podano stanje končno. Naloga preiskovalnega algoritma je bila najti rešitev problema, ki je pot od začetnega do končnega stanja, pri čemer smo pogosto iskali najcenejšo ali najkrajšo pot.

V tem predavanju se osredotočimo na drugačen tip problema: odločanje v igrah z dvema igralcema. Tudi tukaj lahko igro predstavimo kot prostor stanj po katerem se lahko sprehajamo s preiskovalnim drevesom. Bistvena razlika med prostorom stanj za reševanje problemov in prostora stanj za igre je v tem, da se poteze igre, ki so analogne prehodom med stanji, se izmenjujejo med dvema igralcema z nasprotujočimi si interesi (in cilji). Namesto iskanja poti do končnega stanja, nas zanima odgovor vprašanje:

> Katera je optimalna poteza za našega igralca v trenutnem stanju igre?

Optimalna poteza je tista poteza, ki igralcu zagotavlja najboljši možni izid igre ob predpostavki, da tudi nasprotnik igra optimalno. To pomeni, da ne iščemo poteze, ki je dobra zgolj proti šibkemu nasprotniku, temveč potezo, ki je najboljša tudi v najslabšem možnem scenariju, torej ob (za nas) najbolj neugodnem odzivu nasprotnika. Pristopi predstavljeni v tem poglavju slonijo na nekaj predpostavkah:

  * V igri sodelujeta dva igralca in imata v vsakem trenutku oba popolno in enako informacijo o trenutnem stanju igre (temu rečemo igra s _popolno informacijo_);
  * Igra je deterministična in ne sloni na naključjih (izključene so torej skoraj vse igre s kartami ali igralnimi kockami);
  * Oba igralca sta racionalna in se vsak zase trudi doseči najboljši možni končni izid igre;
  * Izid igre je tak, da zmaga enega igralca avtomatično pomeni poraz nasprotnika (to ne izključuje možnosti neodločenega izida). Taki igri rečemo tudi igra z ničelno vsoto (angl. _zero-sum game_). Vrednost končnega izida igre z ničelno vsoto lahko (z vidika izbranega, _našega_ igralca) izrazimo z eno samo številko (na primer, +1 za zmago, 0 za neodločen izid, in -1 za poraz).

Če gre za igro z ne-ničelno vsoto ali tako, ki vključuje več igralcev, situacija postane bistveno bolj zapletena in zahteva drugačne pristope, ki presegajo uvodne vsebine iz umetne inteligence.

V okviru predavanja bomo spoznali vsebino šestega poglavja z naslovom _Adverssaliral Search and Games_ iz učbenika {cite}`russell2019aima`. V okviru predavanja se bomo omejili na razdelke 6.1 do 6.3.

## Igra z ničelno vsoto in igralna strategija

Definirajmo zdaj bolj formalno pojme iz uvodnih odstavkov. Poglejmo najprej definicijo deterministične igre za dva igralca s popolno informacijo.

```{prf:definition} Deterministična igra za dva igralca s popolno informacijo
:label: def-igra-dva-igralca-popolna-informacija

Igra $\mathcal{I}$ za dva igralca s popolno informacijo je sedmerica $\left< S, s_0, S_K, A, P, U_1, U_2 \right>$, kjer je:
  - $S$ _množica stanj_ (rečemo jim tudi pozicije) igre;
  - $s_0 \in S$ je začetno stanje igre, ko je na potezi prvi igralec;
  - $S_K \subseteq S$ je množica končnih stanj igre, ko je znan njen _izid_;
  - $A(s)$ je množica možnih _potez_ igre v podanem stanju $s \in S$;
  - $P(s, p)$ je _funkcija prehoda_, ki je za podano stanje igre $s \in S$ in potezo $p$ vrne stanje igre, ki nastane po odigrani potezi $p$;
  - $U_1(s_k)$ in $U_2(s_k)$ sta _funciji koristnosti_, ki vrneta vrednost izida igre v končnem stanju $s_k \in S_K$ za prvega in drugega igralca.
```

Primerjaj to definicijo z definicijo prostora stanj iz prejšnjega predavanja. Katere razlike opažaš? Premisli, kakšna je množica $A(s_k)$ za podano končno stanje $s_k \in S_K$.

Zdaj lahko definiramo še posebno kategorijo iger iz zgornje definicije, ki jih bomo poimenovali igre z ničelno vsoto.

```{prf:definition} Igra z nicelno vsoto
:label: def-igra-nicelna-vsota

Naj bo $\mathcal{I} = \left< S, s_0, S_K, A, P, U_1, U_2 \right>$ deterministična igra za dva igralca s popolno informacijo. To je igra z ničelno vsoto, če za vsako končno stanje $s_k \in S_K$ velja
$$ U_1(s_k) + U_2(s_K) = 0.$$

V tem primeru lahko igro opišemo z eno samo funkcijo koristnosti $U \colon S_K \to \mathbb{R}$, definirano s predpisom $ U(s_k) = U_1(s_k) = - U_2(s_k)$.

Vrednost te funcije interpretiramo kot vrednost izida igre z vidika prvega igralca, ki ga imenujemo MAX. Drugega igralca igre z ničelno vsoto imenujemo MIN. Igro z ničelno vsoto $\mathcal{I}$ definira šesterica $\left< S, s_0, S_K, A, P, U \right>$, ker imamo eno funkcijo koristnosti $U$.
```

V igri z ničelno vsoto je torej zmaga igralca MAX enaka porazu igralca MIX in obratno. Primer take igre je križec-krožec, kjer dva igralca igrata na kvadratni plošči dimenzij $3 \times 3$ z devetimi polji, ki so na začetku prazna. V vsaki potezi igralec, ki je na vrsti za potezo, izbere eno izmed praznih polj na plošči in v njem postavi svoj znak: prvi igralec uporablja križce (X, zato ga imenujemo tudi igralec X), drugi krožce (O, igralec O). Igra se konča, ko so vsa polja zapolnjena z znaki X in O ali pa je eden od igralcev uspel postaviti tri svoje znake v vrsto treh zaporednih polj. Vrsta je lahko horizontalna, vertikalna ali ena izmed obeh diagonal igralne plošče. Vrednost končnega stanja $s_k \in S_K$ igre križec-krožec poda naslednja funkcija koristnosti:
$$ U(s_k) = \begin{cases}
    +1 & \text{če zmaga prvi igralec MAX (X)} \\
    0 & \text{če je plošča polna in noben igralec ni sestavil vrste treh zaporednih polj s svojimi znaki} \\
    -1 & \text{če zmaga drugi igralec MIN (O)}
\end{cases}. $$

Premisli kako formalno definiramo ostalih pet elementov šesterice $\left< S, s_0, S_K, A, P, U \right>$ za igro križec-krožec.

```{prf:definition} Igralna strategija in optimalna igralna strategija
:label: def-igralna-strategija

Igralna strategija igralca v igri $\mathcal{I} = \left< S, s_0, S_K, A, P, U \right>$ je funkcija $\sigma \colon S \to A(S)$, ki vsakemu stanju $s \in S$, kjer je opazovani igralec na potezi, priredi eno izmed dovoljenih potez $p \in A(S)$.

Igralna strategija $\sigma$ je optimalna, če opazovanemu igralcu zagotavlja najbolj ugoden izid igre v primeri poljubne, tudi optimalne igralne strategije nasprotnika.
```

Premisli kaj je optimalna igralna strategija za igralca X v igri križec-krožec.

## Igralna strategija MINIMAX

Vpeljimo zdaj funkcijo $V$, ki posploši funkcijo koristnosti iz množice končnih stanj na poljubno stanje igre. Vrednost funkcije $V$ za podano stanje bomo poimenovali vrednost stanja.

```{prf:definition} Vrednost stanja MINIMAX
:label: def-vrednost-MINIMAX

Za igro $\mathcal{I} = \left< S, s_0, S_K, A, P, U \right>$ naj bo funkcija $V \colon S \to \mathbb{R}$ definirana z rekuzivnim predpisom
$$ V(s) = \begin{cases}
    U(s) & \text{če} \; s \in S_K \\
    \max_{p \in A(s)} V(P(s, p)) & \text{če je na potezi igralec MAX} \\
    \min_{p \in A(s)} V(P(s, p)) & \text{če je na potezi MIN}
\end{cases}.$$

Funkcijo $V$ imenujemo funkcijo MINIMAX, njeno vrednost $V(s)$ za podano stanje $s \in S$ imenujemo vrednost MINIMAX stanja $s$.
```

Definirana vrednost stanja igre nam omogoča zasnovati optimalno igralno strategijo MINIMAX.

```{prf:definition} Igralna strategija MINIMAX
:label: def-strategija-MINIMAX

Za igro $\mathcal{I} = \left< S, s_0, S_K, A, P, U \right>$ naj bo $V$ funkcija MINIMAX. Igralna strategija MINIMAX za prvega igralca MAX v stanju igre $s \in S$ izbere potezo iz množice
$$ \left\{ p^{*} \colon V(P(s, p^{*})) = \max_{p \in A(s)} V(P(s, p)) \right\}, $$
za igralca MIN pa MIN pa potezo iz množice
$$ \left\{ p^{*} \colon V(P(s, p^{*})) = \max_{p \in A(s)} V(P(s, p)) \right\}. $$
```

Zdaj bomo dokazali, da je igralna strategija MINIMAX optimalna.

```{prf:theorem} Optimalnost igralne strategije MINIMAX
:label: thm-optimalnost-MINIMAX

Naj bo igra $\mathcal{I} = \left< S, s_0, S_K, A, P, U \right>$ za dva igralca deterministična, končna (ima končno množico stanj $S$) in z ničelno vsoto. Vrednost funkcije MINIMAX za vsako stanje $s \in S$ pravilno izračuna vrednost igre $\mathcal{I}$ ob predpostavki optimalne igre obeh igralcev.
```

```{prf:proof} Optimalnost igralne strategije MINIMAX
:label: prf-optimalnost-MINIMAX

Dokazujemo po globini poddrevesa rekurzivnega izračuna funkcije MINIMAX s korenom v stanju $s$.

1. Osnovni korak za poddrevo globine $0$.

    Če je globina poddrevesa v stanju $s$ enaka $0$, pomeni, da smo v končnem stanju igre, $s \in S_K$. Algoritem za izračun funkcije MINIMAX vrne v tem primeru vrednost $U(s_k)$, ker je prava vrednost izida igre v stanju $s$ in je izbira trivialno optimalna.

1. Predpostavimo, da igralna strategija MINIMAX ponudi optimalno strategijo za vsa stanja $s$ z globino izračuna $\le d$. Dokazali bomo, da MINIMAX izbere optimalno potezo za vsa stanja $s$ z globino izračuna $d+1$.

    * Če je na potezi igralec MAX, strategija MINIMAX izbere potezo, ki pelje k največji možni vrednosti izida med vsemi možnimi naslednjimi stanji s poddrevesi globine največ $d$. Po induktivni predpostavki so vse poteze v teh poddrevesih izbrene optimalno, zato je tudi ta izbor optimalen.

    * Če je na potezi nasprotnik MIN, strategija MINMAX izbere potezo, ki pelje k najmanjši možni vrednosti izida igre (najslabšem možnem scenariju za igralca MAX) med vsemi poddrevesi globine največ $d$. Taka izbira je ekvivalentna predpostavki, da nasprotnik MIN igra optimalno. 
```

V naslednjem razdelku bomo najprej zasnovali algoritem za izračun funkcije vrednosti stanja MINIMAX in nato še algoritem, ki implementira igralno strategijo MINIMAX.

## Algoritem MINIMAX

Algoritem {ref}`alg-minimax` izračuna vrednost MINIMAX za poljubno podano stanje igre po rekurzivnem predpisu iz definicije. Rekurzivni izračun vrednosti funkcije MINIMAX po tem algoritmu razpne drevo izračuna s korenskim vozliščem v začetnem stanju igre (privzeta vrednost argumenta `s`, glej vrstici 3 in 10) in končnimi vozlišči v končnih stanjih igre, ker se izračun konča z robnim pogojem rekurzije, ki izenači vrednost MINIMAX z vrednostjo funkcije koristnosti v tem stanju (vrstica 13).

```{code-block} text
:caption: Osnovni algoritem MINIMAX z omejeno globino
:name: alg-minimax
:linenos:

VHOD:
    igra      # definicija igre
    s         # trenutno stanje igre, privzeta vrednost je začetno stanje
    globina   # preostala globina iskanja, privzeta vrednost je ∞
    max       # True: na potezi je MAX; False: na potezi je MIN

IZHOD:
    vmm       # vrednost MINIMAX podanega stanja igre s

FUNCTION MINIMAX(igra, s = igra.zacetno_stanje, globina = ∞, max = True):

    if globina = 0 or igra.koncno(s) then
        return igra.U(s) oziroma igra.h(s)

    if max then
        vmm ← -∞
        for poteza in igra.A(s) do
            vmm ← max(vmm, MINIMAX(igra, P(s, poteza), globina - 1, False))
    else
        vmm ← +∞
        for poteza in igra.A(s) do
            vmm ← min(vmm, MINIMAX(igra, P(s, poteza), globina - 1, True))
    return vmm
```

Iz praktičnih razlogov algoritem omeji globino drevesa izračuna z ustrezno nastavitvijo argumenta (ki ima privzeto vrednost `∞`, glej vrstici 4 in 10). Zato se v vrstici 12 odločimo, da vrnemo vrednost funkcije MINIMAX tudi takrat, ko smo dosegli podano globino drevesa. Če torej poganjamo algoritem z omejeno globino, se lahko zgodi, da moramo vrniti vrednost stanja igre `s`, ki ni končno. V tem primeru algoritem nima na voljo vrednosti funkcije koristnosti `U`, zato vrne hevristično oceno `h(s)` vrednosti stanja `s`. O hevristični oceni stanje igre bomo spregovorili v naslednjem razdelku.

V nadaljevanju definicije funkcije MINMAX (vrstice 15-22) uporabimo rekurzivne klice za izračun njene vrednosti v skladu z rekurzivnim predpisom iz definicije {ref}`def-vrednost-MINIMAX`.

Slika {ref}`fig-minimax-primer` prikazuje primer drevesa izračuna vrednosti MINIMAX za namišljeno igro z dvema igralcema. Rekurzija se sprehodi po vozliščih tega drevesa v globino. Začne v vozlišču številka 1 in se po najbolj levi veji (1, 4, 8 in 16) spusti do lista 16, kjer dobi vrednost funkcije MINIMAX 9. Nato se vrne v vozlišče 8 in nastavi MINIMAX tega vozlišča na 9. V naslednjem koraku algoritem obišče list 17 z vrednostjo 8 in ob povratku v vozlišče 8 spremeni vrednosti njegove funkcije MINIMAX na 8, ker je 8 manjše od (trenutne) vrednosti 9. Nato se vrne v vozlišče 4, tam nastavi vrednost funkcije MINIMAX na 8 in nadaljuje pot po veji čez vozlišče 9 do vozlišča 18.

Za vajo se sprehodi po celotnem drevesi in premisli kako so izračunane vrednosti funkcije MINIMAX v vseh vozliščih.

```{figure} ../materiali/minimax-primer.png
---
name: fig-minimax-primer
---
Primer drevesa rekurzivnega izračuna vrednosti funkcije MINIMAX. Trikotniki obrnjeni navzgor ustrezajo potezam igralca MAX, tisti obrnjeni navzdol pa igralcu MIN. Številka v trikotniku je enaka vrednosti funkcije MINIMAX v vozlišču. Številka zraven trikotnika je številka vozlišča.
```

Nadaljujmo zdaj z izračunom MINIMAX za drevo na Sliki{ref}`fig-minimax-primer` v vozlišču 18, kjer smo se prej ustavili. V tem vozlišču je vrednost funkcije MINIMAX enaka 6, vrnemo se en korak nazaj in nastavimo vrednost 6 tudi v vozlišču 9. Običajno bi bilo, da sedaj nadaljujemo tako, da obiščemo še drugega otroka vozlišča 9, t.j., vozlišče 19. Ampak ta obisk, ne glede na vrednost funkcije MINIMAX v tem vozlišču, nam ne more _povečati_ trenutne (minimalne) vrednosti 6 v vozlišču 9. Ker je vrednost 6 že manjša od trenutne vrednosti 8 v vozlišču 4 (kjer iščemo maksimum), obisk vozlišča 19 nima smisla. Rečemo, da lahko poddrevo desno od vozlišča 18 odrežemo.

Ta premislek velja tudi za druga vozlišča v drevesu izračuna funkcije MINIMAX. Pravzaprav gre za splošno pravilo rezanja nepotrebnih poddreves, t.j., poddreves, ki ne vplivajo na vrednost funkcije MINIMAX. To splošno pravilo lahko uporabimo za bolj učinkovito verzijo algoritma MINIMAX. Algoritem {ref}`alg-minimax-rezanje` predstavi rezanje ALPHABETA, ki lahko drastično zmanjša (poreže) drevo rekurzivnih izračunov funkcije MINIMAX. Poudarjene so vrstice, ki so bile spremenjene in/ ali dodane originalni različici algoritma MINIMAX.

```{code-block} text
:caption: Algoritem MINIMAX z rezanjem ALPHABETA
:name: alg-minimax-rezanje
:linenos:
:emphasize-lines: 6,11,19-21,25-27

VHOD:
    igra        # definicija igre
    s           # trenutno stanje igre
    globina     # preostala globina iskanja, privzeta vrednost je ∞
    max         # True: na potezi je igralec MAX; False: na potezi je MIN
    alpha, beta # meje rezanja, privzeti vrednosti alpha=-∞ in beta=+∞

IZHOD:
    vmm         # vrednost MINIMAX podanega stanja igre s

FUNCTION minimax(igra, s = igra.zacetno_stanje, globina = ∞, max = True, alpha=-∞, beta=+∞):

    if globina = 0 or igra.koncno(s) then
        return igra.U(v)

    if max then
        vmm ← -∞
        for poteza in igra.A(s) do
            vmm ← max(vmm, MINIMAX(igra, P(s, poteza), globina - 1, False, apha, beta))
            alpha ← max(alpha, vmm)
            if alpha ≥ beta then break 
    else
        vmm ← +∞
        for poteza in igra.A(s) do
            vmm ← min(vmm, MINIMAX(igra, P(s, poteza), globina - 1, False, alpha, beta))
            beta ← min(beta, vmm)
            if alpha ≥ beta then break 
    return vmm
```

Rezanje deluje tako, da v vsakem vozlišču drevesa izračunamo dve dodatni vrednosti:

  * $\alpha$ predstavlja *najvišjo** vrednost funkcije MINIMAX, ki si jo je igralec MAX doslej zagotovil v trenutnem stanju igre (glej vrstice 19-21);

  * $\beta$ predstavlja *najnižjo* vrednost funkcije MINIMAX, ki si jo je igralec MIN doslej zagotovil (glej vrstice 25-27).

Če v algoritem v trenutnem vozlišču ugotovi, da velja $\alpha \geq \beta$, vemo, da nadaljnje pregledovanje preostalih otrok trenutnega vozlišča ne more več vplivati na vrednost funkcije MINIMAX v tem vozlišču. Zato veje do teh otrok lahko **odrežemo**.

Za vajo se sprehodi po primeru drevesa izračuna iz slike {ref}`fig-minimax-primer` in ugotovi katere veje tega drevesa bo algoritem ALPHABETA porezal. V pomoč pri tem je drevo izračuna iz slike {ref}`fig-alphabeta-primer` s končnimi vrednostmi $\alpha$ in $\beta$ v vseh vozliščih.

```{figure} ../materiali/alphabeta-primer.png
---
name: fig-alphabeta-primer
---
Primer drevesa rekurzivnega izračuna vrednosti funkcije MINIMAX s končnimi vrednostmi $\alpha$ in $\beta$, ki jih izračuna rezanje ALPHABETA. Rdeča barva za vrednost funkcije MINIMAX pomeni, da algoritem z rezanjem ni izračunal te vrednosti.
```

## Hevristična ocena vrednosti stanja

Vrnimo se zdaj primeru preiskovanja MINIMAX z omejeno globino, kjer moramo ovrednotiti podano stanje igre $s$, ki ni končno, torej $s \notin s_K$. Ker za tako stanje funkcija koristnosti ni definirana, potrebujemo hevristično funkcijo, ki oceni približno vrednost funkcije MINIMAX podanega stanja brez rekurzivnega razvejanja vseh možnih potez iz tega stanja. Podobno kot pri običajnih hevrističnih funkcijah iz prejšnjega predavanja, tudi tukaj si želimo, da je po eni strani vrednost funkcije čim boljši približek pravi vrednosti MINIMAX za to vozlišče. Po drugi strani si pa želimo, da bi bila hevristika hitro in učinkovito izračunljiva. Seveda pa približek funkcije MINIMAX tudi pomeni, da pri uporabi hevristike nimamo več garancije optimalne igralne strategije.

Veliko premisleka, spretnosti in poizkusov potrebujemo za snovanje ustrezne hevristike, ki bi podala dober približek točni vrednosti MINIMAX v kateremkoli stanju igre. Najbolj običajna zasnova hevristik je linearna kombinacija značilk $f_i$:
$$h(s) = w_1 f_1(s) + w_2 f_2(s) + \dots + w_n f_n(s).$$

Vsaka značilka je funkcija $f_i \colon S \to \mathbb{R}$ trenutnega stanja igre $s$ in vsaki značilki priredimo utež $w_i \in \mathbb{R}$. Značilke so katerekoli lastnosti podanega stanja igre $s$, ki jih lahko enostavno izračunamo in imajo numerično vrednost. Na primer, pri igranju šaha, lahko opazujemo število različnih figur igralca MAX in igralca MIN: za vsako šahovsko figuro lahko opazujemo razliko med številom figur prvega in drugega igralca. V takem primeru so uteži pozitivna števila, saj prednost pri določenem številu figur lahko pomeni "boljše" stanje, ki lahko prinese zmago prvemu igralcu (MAX).

Linearna oblika hevristik poenostavi njihovo ročno zasnovo, a nikakor ni predpisana. Hevristika je lahko tudi nelinearna funkcija, ki se jo iz prejšnjih partij naučimo s strojnim učenjem (ki ga bomo spoznali v drugem delu semestra). Pomembno je imeti v mislih le to, da je dobra hevristika tista, ki z visokimi vrednostmi ovrednoti tista stanja, kjer je prvi igralec MAX bližji boljšemu izidu igre (in obratno, nizke vrednosti za stanja, kjer je nasprotnik MIN bližje zmagi). To pogosto zahteva preizkušanje velikega števila različnih funkcij in opazovanja obnašanja algoritma MINIMAX.

# Naloga za bonus točke

1. Implementiraj algoritem MINIMAX z rezanjem ALPHABETA in ga uporabi na igri križec-krožec. Premisli o tem s katero hevristiko bi lahko omejil globino drevesa izračuna vrednosti MINIMAX za to igro.

<br/><br/>